# CN7 01_preprocess_train
지우 final e8e354f 원본을 사용한다. Python3.11 가상환경 활성화 후 cn7/requirements.txt 설치. 모델의 통합 이상 경보는 가스/미성형 유형 분류가 아니다. 탐지와 확률층 개발평가를 새 독립성능으로 해석하지 않는다.


In [ ]:
from pathlib import Path
import sys,os
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'cn7/vendor/jiwoo_final').exists())
VENDOR=ROOT/'cn7/vendor/jiwoo_final'
sys.path.insert(0,str(VENDOR/'src'))
import paths
paths.ROOT=ROOT
paths.DATA=ROOT/'dataset/runtime'
paths.OUT=ROOT/'cn7/outputs'
for name,rel in {'INTER':'intermediate','LINKED':'intermediate/linked','BASE':'intermediate/base','KAMP_CACHE':'intermediate/shots_cache.pkl','W_PRIOR':'intermediate/w_prior.json','FINAL':'final_model','PROB':'prob','LEDGER':'ledger','PRED':'pred','EXP':'experiments'}.items():
 setattr(paths,name,paths.OUT/rel)
paths.ensure_dirs()
os.environ['KAPPA']='10'

import gzip,shutil,json,hashlib,runpy
paths.DATA.mkdir(parents=True,exist_ok=True)
for name in ['labeled_data.csv','moldset_labeled.csv']:
 shutil.copy2(ROOT/'dataset/original'/name,paths.DATA/name)
for name in ['moldset_labeled_cn7.csv','moldset_labeled_rg3.csv']:
 shutil.copy2(ROOT/'dataset/kamp'/name,paths.DATA/name)
with gzip.open(ROOT/'dataset/original/unlabeled_data.csv.gz','rb') as src,(paths.DATA/'unlabeled_data.csv').open('wb') as dst:
 shutil.copyfileobj(src,dst)
paths.check_data()
runner=runpy.run_path(str(VENDOR/'run_all.py'),run_name='cn7_package')
for no,name in enumerate(['s1_prep','s2_model','s3_prob','s4_ledger','s5_audit','s6_compare','s7_ops','s8_pred'],1):
 print('Step',no,name,flush=True)
 runner[name]()
assert runner['s9_check'](),'지우 원본 재현 확인 실패'
processed=ROOT/'dataset/processed/cn7';processed.mkdir(parents=True,exist_ok=True)
shutil.copy2(paths.BASE/'cn7_shots.csv',processed/'cn7_shots.csv')
shutil.copy2(paths.LINKED/'linked_unlabeled_cn7.csv',processed/'linked_unlabeled_cn7.csv')
models=ROOT/'cn7/models';models.mkdir(exist_ok=True)
shutil.copy2(paths.FINAL/'cn7_final_model.pkl',models/'cn7_final_model.pkl')
import final_model as fm
model=fm.FinalModel.load(str(models/'cn7_final_model.pkl'))
reference,log=fm.build_reference('cn7',fm.load_labeled_shots('cn7'),period=model.ref_period)
reference_features=fm.make_features(reference,model.steps,model.fr_step,model.steps_all)
reference.to_csv(processed/'cn7_normal_reference_raw.csv',index=False,encoding='utf-8-sig')
reference_features.to_csv(processed/'cn7_normal_reference_features.csv',index=False,encoding='utf-8-sig')
print('Saved CN7 model:',models/'cn7_final_model.pkl')
print('CN7 통합 이상 경보이며 가스/미성형 유형 분류는 아님')

